# GRPO full training: Qwen2.5-3B-Instruct + bf16 LoRA + vLLM on MedQA

Pilot takeaways are in `results_summary/grpo/pilot/bf16_vllm_a100/README.md`: bf16 LoRA + vLLM on an A100 takes about 8.6 s per step (512-token answer limit); **don't use QLoRA + vLLM** (the gradients fade out).

Training data: the **5000 training questions** in `data/medqa_split_seed42.json` (the other 500 validation questions aren't used for training; SFT / DPO use the same split). 8 answers per question. **Train twice, with seeds 1 and 2**: after finishing `SEED = 1`, change `SEED` in cell 2 to 2 and run everything again (it writes to a different output folder).

Run in order: check GPU -> set variables -> pull the repo -> install dependencies -> mount Drive -> (optional) HF_TOKEN -> unit tests -> **train** -> look at results

Before running: **Runtime -> Change runtime type -> A100 GPU**.

**All output (checkpoints, metrics.csv, train_log.txt, final adapter) goes straight to `OUTPUT_DIR` on Google Drive**, so a disconnect doesn't lose anything.

**Resuming after a disconnect / the runtime getting recycled**: rerun cells 1 through 7 (this reinstalls dependencies), then run cell 8 **as is**. The command has `--resume`, so it picks up from the latest checkpoint in `OUTPUT_DIR/checkpoints`, and the log shows `[resume] found checkpoint`.

**Run 300 steps first, then decide**: with `STOP_AFTER_STEP = 300` in cell 2, cell 8 saves a checkpoint at step 300 and stops (the learning rate still decays on the 1250-step schedule). After checking KL / reward in cell 9, set `STOP_AFTER_STEP` to `0` and rerun from cell 1 (or just cells 2 and 8 if the runtime is still alive); it continues from checkpoint-300 to the end. `STOP_AFTER_STEP` is the only setting you can change when resuming.

Heads up: when resuming, **don't change anything else in cell 2** (number of questions, epochs, seed, hyperparameters), or the LR schedule and data order won't line up.

In [ ]:
# 1. Check the GPU and Python version
!nvidia-smi
import sys, torch
print('Python', sys.version.split()[0])
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    print('GPU:', torch.cuda.get_device_name(0), f'| compute {major}.{minor} →', 'bf16' if major >= 8 else 'fp16')
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> GPU'
assert sys.version_info >= (3, 11), 'TRL 1.16 needs Python >= 3.11'

In [ ]:
# 2. Set variables (only edit here; don't change them once training starts, or resuming won't line up)
REPO_URL = 'https://github.com/skjdsk/medqa-calibration.git'
BRANCH   = 'main'
REPO_DIR = '/content/medqa-calibration'

TRL_VERSION = '1.16.0'

# training data: the 5000 training questions in data/medqa_split_seed42.json (the 500 validation questions aren't used)
# which questions get used comes from the split file, not SEED; SFT / DPO use the same split file
N_QUESTIONS           = 0      # 0 = all 5000 training questions
SEED                  = 1      # training seed (LoRA init, sampling, data order): run 1 first, then change to 2 for the second run
NUM_EPOCHS            = 1

NUM_GENERATIONS       = 8      # answers per question; 8 gives fewer all-right/all-wrong groups (no gradient) than 4
MAX_COMPLETION_LENGTH = 1024   # with 512 in the pilot, about 9% of answers got cut off
LEARNING_RATE         = 5e-5   # at 1e-5 KL was only ~0.0015 and the validation results were basically the same as base (see notebooks/grpo/grpo_val_compare.ipynb)
BETA                  = 0.04   # KL coefficient
TEMPERATURE           = 1.0
PER_DEVICE_BS         = 4      # if you run out of memory, use 2 and set GRAD_ACCUM to 16 (same answers per step)
GRAD_ACCUM            = 8      # answers per step = 4 x 8 = 32 = 4 questions x 8 answers; has to be a multiple of NUM_GENERATIONS
LORA_R, LORA_ALPHA    = 16, 32
LORA_TARGET_MODULES   = 'all-linear'
USE_VLLM              = True
QUANTIZATION          = 'none' # bf16 + LoRA; don't use '4bit' with vLLM
STOP_AFTER_STEP       = 300    # run only the first 300 steps to check KL / reward; once it looks fine, set to 0 and rerun cell 8 to continue from checkpoint-300
                               # (this one is safe to change: it doesn't affect the LR schedule or data order)
SAVE_STEPS            = 100    # checkpoint every 100 steps (with optimizer state, about 0.36 GB; only the latest 2 are kept)
#   deleted old checkpoints go to the Drive trash and still take up space: empty the trash during or after training

# the output folder is on Drive and its name comes from the settings: same settings -> same folder -> resuming finds the checkpoint
EXP_NAME   = f"grpo_train{N_QUESTIONS or 5000}_g{NUM_GENERATIONS}_ep{NUM_EPOCHS}_len{MAX_COMPLETION_LENGTH}_lr{LEARNING_RATE:g}_seed{SEED}_{'qlora' if QUANTIZATION == '4bit' else 'bf16'}"
OUTPUT_DIR = f'/content/drive/MyDrive/medqa-calibration/grpo/{EXP_NAME}'

assert (PER_DEVICE_BS * GRAD_ACCUM) % NUM_GENERATIONS == 0, 'PER_DEVICE_BS x GRAD_ACCUM has to be divisible by NUM_GENERATIONS'
prompts_per_step = PER_DEVICE_BS * GRAD_ACCUM // NUM_GENERATIONS
import math
n = N_QUESTIONS or 5000
print('per step:', PER_DEVICE_BS * GRAD_ACCUM, 'answers =', prompts_per_step, 'questions x', NUM_GENERATIONS, 'answers')
print('total steps ~', math.ceil(n * NUM_EPOCHS / prompts_per_step))
print('output folder:', OUTPUT_DIR)

In [ ]:
# 3. git clone / pull repo
# Private repo: add GITHUB_TOKEN (a GitHub personal access token) under the key icon (Secrets) on the left in Colab, and turn on "Notebook access"
import os, subprocess
from pathlib import Path

def sh(cmd, shown=None):
    print('$', shown or cmd)
    subprocess.run(cmd, shell=True, check=True)

clone_url = REPO_URL
try:
    from google.colab import userdata
    gh_token = userdata.get('GITHUB_TOKEN')
    if gh_token and REPO_URL.startswith('https://github.com/'):
        clone_url = REPO_URL.replace('https://', f'https://{gh_token}@')
except Exception:
    pass  # no GITHUB_TOKEN set, treat it as a public repo

if (Path(REPO_DIR) / '.git').exists():
    sh(f'git -C {REPO_DIR} fetch origin && git -C {REPO_DIR} checkout {BRANCH} && git -C {REPO_DIR} pull origin {BRANCH}')
else:
    sh(f'git clone --branch {BRANCH} {clone_url} {REPO_DIR}', shown=f'git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}')
os.chdir(REPO_DIR)
sh('git log -1 --oneline')
for f in ['train/grpo/grpo_data.py', 'train/grpo/rewards.py', 'train/grpo/test_rewards.py', 'train/grpo/grpo.py', 'train/grpo/estimate.py', 'data/medqa_split.py', 'data/medqa_split_seed42.json']:
    assert Path(f).exists(), f'{f} not found in the repo: make sure it has been pushed to {BRANCH}'
print('cwd =', os.getcwd())

In [ ]:
# 4. Install dependencies
from pathlib import Path
ip = get_ipython()
if Path('requirements.txt').exists():
    ip.run_line_magic('pip', 'install -q -r requirements.txt')
ip.run_line_magic('pip', f'install -q "trl[peft,quantization]=={TRL_VERSION}" "matplotlib" "pandas" "pytest"')
if USE_VLLM:
    # vLLM installs the torch version it wants, so you usually need Runtime -> Restart session afterwards, then rerun from the "set variables" cell
    # vLLM 0.28+ dropped bitsandbytes quantization, but with QLoRA (4-bit) TRL has vLLM load the model through bitsandbytes
    # -> pinned to 0.27.1 (the last version with bitsandbytes, inside TRL 1.16's allowed 0.22-0.31 range, needs torch==2.13.0)
    ip.run_line_magic('pip', f'install -q "trl[vllm]=={TRL_VERSION}" "vllm==0.27.1"')

# Colab's preinstalled torchao 0.10 is too old: peft 0.21 errors out when adding LoRA without quantization (--quantization none)
# "only versions above 0.16.0 are supported". We don't use torchao (neither vllm nor peft needs it), so just uninstall it
ip.system('pip uninstall -y -q torchao')

import importlib.metadata as md
for pkg in ['torch', 'transformers', 'trl', 'peft', 'bitsandbytes', 'accelerate', 'datasets'] + (['vllm'] if USE_VLLM else []):
    try:
        print(f'{pkg:14s}', md.version(pkg))
    except md.PackageNotFoundError:
        print(f'{pkg:14s} not installed')
print('\nIf pip says a restart is needed above (or imports fail later): Runtime -> Restart session, then rerun from the "set variables" cell.')

In [ ]:
# 5. Mount Google Drive and check the output folder for earlier checkpoints
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('output folder:', OUTPUT_DIR)
ckpts = sorted(Path(OUTPUT_DIR, 'checkpoints').glob('checkpoint-*'), key=lambda p: int(p.name.split('-')[-1]))
if ckpts:
    print('existing checkpoints:', [p.name for p in ckpts], '-> cell 8 will continue from', ckpts[-1].name)
else:
    print('no checkpoint yet -> cell 8 will train from scratch')
if Path(OUTPUT_DIR, 'adapter').exists():
    print('Heads up: this folder already has a finished adapter. These settings may already be done, and running again will overwrite it')

In [ ]:
# 6. (Optional) set HF_TOKEN: Qwen2.5 and MedQA are public so downloads work without it, but with it they're more reliable and less likely to get rate-limited
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
        print('HF_TOKEN set')
except Exception:
    print('No HF_TOKEN found (optional), downloading anonymously')

In [ ]:
# 7. Run the reward-function unit tests (a few seconds, no GPU)
!python -m pytest -q train/grpo/test_rewards.py

In [ ]:
# 8. Full training (results go straight to OUTPUT_DIR on Drive; after a disconnect just rerun this cell as is)
args = [
    f'--n_questions {N_QUESTIONS}', f'--seed {SEED}', f'--num_train_epochs {NUM_EPOCHS}',
    f'--num_generations {NUM_GENERATIONS}', f'--max_completion_length {MAX_COMPLETION_LENGTH}',
    f'--learning_rate {LEARNING_RATE}', f'--beta {BETA}', f'--temperature {TEMPERATURE}',
    f'--per_device_train_batch_size {PER_DEVICE_BS}', f'--gradient_accumulation_steps {GRAD_ACCUM}',
    f'--lora_r {LORA_R}', f'--lora_alpha {LORA_ALPHA}', f'--lora_target_modules {LORA_TARGET_MODULES}',
    f'--quantization {QUANTIZATION}', f'--output_dir {OUTPUT_DIR}',
    f'--save_steps {SAVE_STEPS}', f'--stop_after_step {STOP_AFTER_STEP}', '--resume',
]
if USE_VLLM:
    args.append('--use_vllm')
ARGS = ' '.join(args)
print('python -m train.grpo.grpo', ARGS)
# tee -a: when resuming, append to the log instead of overwriting it
!python -u -m train.grpo.grpo {ARGS} 2>&1 | tee -a {OUTPUT_DIR}/train_log.txt

**What to check in the first 10 steps** (before walking away):
- `mem`: no out-of-memory
- time per step x total steps ~ total time
- `sampling/importance_sampling_ratio/mean` near 1, `sampling/sampling_logp_difference/mean` around 0.02 and not climbing
- `grad_norm` around 0.05-0.3 (now and then near 0 because every answer in the group was right or wrong)
- `truncated` lower than the pilot's 0.09

**During training** (`OUTPUT_DIR/metrics.csv` gets a row every step, you can open it right in Drive): reward roughly going up, format rate staying around 0.9, KL creeping up but staying under about 0.1, answer length not steadily growing or suddenly dropping.

In [ ]:
# 9. Look at results (run after training. To check progress mid-run, open OUTPUT_DIR/metrics.csv in Drive,
#    don't stop cell 8: you can resume, but you lose the steps since the last checkpoint)
import json
import pandas as pd
from IPython.display import Image, display

df = pd.read_csv(f'{OUTPUT_DIR}/metrics.csv')
cols = ['step', 'step_time_s', 'peak_mem_allocated_gb', 'reward_mean', 'frac_groups_all_same',
        'format_rate', 'mean_completion_tokens', 'truncated_frac', 'kl', 'grad_norm']
print(f'trained {int(df.step.max())} steps')
# average every 50 steps to see the trend
df['block'] = (df.step - 1) // 50
display(df.groupby('block')[[c for c in cols if c in df.columns and c != 'step']].mean().round(4))

report_path = Path(OUTPUT_DIR, 'pilot_report.json')
if report_path.exists():  # only there once training is done
    report = json.load(open(report_path))
    print('avg time per step (s):', report['timing']['avg_step_time_s_excl_warmup'])
    print('peak memory (GB):', report['memory']['peak_mem_allocated_gb_max_over_steps'])
    print('reward:', report['reward'])
    print('format rate:', report['format_rate'])
    print('stability:', report['stability'])
    display(Image(f'{OUTPUT_DIR}/curves.png'))
    print('LoRA adapter：', Path(OUTPUT_DIR, 'adapter'))